# **Fashion MNIST**

In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data= datasets.FashionMNIST(
  root="data", train=True, download=True, transform=ToTensor()
)
test_data= datasets.FashionMNIST(
  root="data", train=False, download=True, transform=ToTensor()
)

In [ ]:
img, label= training_data[0]
print(f'{label= }')
print(f'{type(img) = }')
print(f'{img.shape= }')
print(f'{len(training_data)}')

label= 9
type(img) = <class 'torch.Tensor'>
img.shape= torch.Size([1, 28, 28])
60000


# **DataLoader**

In [ ]:
from torch.utils.data import DataLoader
batch_size= 64

train_dataloader = DataLoader(training_data, batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size, shuffle=False)

In [ ]:
import torch
device= ("cuda"
if torch.cuda.is_available()
else"cpu"
)
print(f"Using{device}device")

Usingcudadevice


# **torch.nn.Flatten**


In [ ]:
from torch import nn

m = nn.Flatten()

for X, y in train_dataloader:
  print(f"Shapeof X [N, C, H, W]: {X.shape}")
  print(f"{m(X).shape = }")
  print(f"Shapeof y: {y.shape}{y.dtype}")
  break

Shapeof X [N, C, H, W]: torch.Size([64, 1, 28, 28])
m(X).shape = torch.Size([64, 784])
Shapeof y: torch.Size([64])torch.int64


# **torch.nn.Linear(in_features, out_features**

In [ ]:
import torch
from torch import nn

model = nn.Linear(30, 20)
input = torch.randn(128,30)
output = model(input)
print(output.shape)


torch.Size([128, 20])


In [ ]:
class NeuralNetwork2(nn.Module):
    def __init__(self):
      super().__init__()
      self.flatten = nn.Flatten()
      self.linear1 = nn.Linear(28*28, 512)
      self.linear2 = nn.Linear(512, 512)
      self.linear3 = nn.Linear(512, 10)
      self.relu1 = nn.ReLU()
      self.relu2 = nn.ReLU()

    def forward(self, x):
      x= self.flatten(x)
      x= self.relu1(self.linear1(x))
      x= self.relu2(self.linear2(x))
      return self.linear3(x)


nn.Sequential 사용

In [ ]:
class NeuralNetwork3(nn.Module):
    def __init__(self):
      super().__init__()
      self.flatten = nn.Flatten()
      self.linear_relu_stack= nn.Sequential(
      nn.Linear(28*28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10)
  )

    def forward(self, x):
      x = self.flatten(x)
      logits = self.linear_relu_strack(x)
      return logits

In [ ]:
model = NeuralNetwork2().to(device)
optim = SGDMomentum(model.parameters(), lr=1e-3 , momentum=0.75)
criterion = nn.CrossEntropyLoss()

# **train 클래스**

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    size = len(dataloader.dataset)
    n_samples = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Forward
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backprop
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # 진행 상태 출력
        n_samples += len(y)
        if batch % 100 == 0:
            loss = loss.item()
            print(f"loss: {loss:7f} [{n_samples:5d}/{size}]")


In [ ]:
epochs= 5
for t in range(epochs):
    print(f"Epoch{t+1}\n-------------------------------")
    train(train_dataloader, model, criterion , optim)
    test(test_dataloader, model, criterion)
    print("Done!")

Epoch1
-------------------------------
loss: 2.298667 [   64/60000]
loss: 2.368706 [ 6464/60000]
loss: 2.470389 [12864/60000]
loss: 3.238040 [19264/60000]
loss:     nan [25664/60000]
loss:     nan [32064/60000]
loss:     nan [38464/60000]
loss:     nan [44864/60000]
loss:     nan [51264/60000]
loss:     nan [57664/60000]
Test Error: 
 Accuracy: 10.0%, Avg loss:      nan 

Done!
Epoch2
-------------------------------
loss:     nan [   64/60000]
loss:     nan [ 6464/60000]
loss:     nan [12864/60000]
loss:     nan [19264/60000]
loss:     nan [25664/60000]
loss:     nan [32064/60000]
loss:     nan [38464/60000]
loss:     nan [44864/60000]
loss:     nan [51264/60000]
loss:     nan [57664/60000]
Test Error: 
 Accuracy: 10.0%, Avg loss:      nan 

Done!
Epoch3
-------------------------------
loss:     nan [   64/60000]
loss:     nan [ 6464/60000]
loss:     nan [12864/60000]
loss:     nan [19264/60000]
loss:     nan [25664/60000]
loss:     nan [32064/60000]
loss:     nan [38464/60000]
loss:  

In [ ]:
def test (dataloader , model , loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  loss_sum, correct= 0, 0
  with torch.no_grad():
    for x ,y in dataloader:
      x, y= x.to(device), y.to(device)
      pred = model(x)
      loss = loss_fn(pred , y)
      loss_sum += loss.item() * len(y)
      correct += (pred.argmax(dim=1) ==y).sum().item()
    loss_sum /= size
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {loss_sum:>8f} \n")


# **Stable Softmax**

In [ ]:
def softmax_naive(x):
  ex = torch.exp(x)
  print(ex)
  print(ex.sum(dim = -1,keepdim=True))
  return ex / ex.sum(dim = -1,keepdim=True)

In [ ]:
def softmax_stable(x):
  z , _ = x.max(dim=-1 , keepdim=True)
  print(f'z: {z}')
  print(f'_:{_}')
  ex = torch.exp(x - z)
  return ex / ex.sum(dim=-1, keepdim=True)

In [ ]:
# Case 1: Small numbers
import torch
x_small= torch.tensor([1.0, 2.0, 3.0])
print("Naive:", softmax_naive(x_small))
print("Stable:", softmax_stable(x_small))
# Case 2: Large numbers
x_large= torch.tensor([1000.0, 1001.0, 1002.0])
print("Naive:", softmax_naive(x_large)) # Expect NaN/ inf
print("Stable:", softmax_stable(x_large)) # Works fine!

In [ ]:

train_dataloader= DataLoader(training_data, batch_size=128)
def train(dataloader, model, loss_fn, optimizer):
  model.train()
  for x, y in dataloader:
      y_pred= model(x)
      loss= loss_fn(y_pred, y)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

# **128의 배치사이즈가 너무 커서 GPU에 다 안들어갈때는**

In [ ]:
train_dataloader= DataLoader(training_data, batch_size=32)
def train(dataloader, model, loss_fn, optimizer):
  i = 0
  model.train()
  for x, y in dataloader:
      y_pred= model(x)
      loss = loss_fn(y_pred, y)
      loss.backward()
      i+=1
      if (i % 4 ==0):
        optimizer.step()
        optimizer.zero_grad()

# ***Optimizer prectice***

In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Dict, Iterable
import torch
class Myoptimizer(ABC):
  def __init__(self, params:Iterable[torch.nn.Parameter] , lr:float):
    self.params = list(params)
    self.lr = lr

  def zero_grad(self) ->None:
    for p in self.params:
      p.grad = None

  @torch.no_grad() #python decorator
  def step(self) -> None:
    for p in self.params:
      if p.grad is None:
        continue
      self._update_pram(p)

  @abstractmethod
  def _update_pram(self, p:torch.nn.parameter) -> None:
    pass

In [ ]:
class MySGD(Myoptimizer):
  def _update_pram(self, p:torch.nn.parameter) -> None:
    p -= self.lr * p.grad

class MyOptimwithState(Myoptimizer):
  def __init__(self, params:Iterable[torch.nn.Parameter] , lr:float):
    super().__init__(params , lr)
    self._state = Dict[torch.nn.Parameter , Dict[str , Any]] = {}

  def state(self, p:torch.nn.parameter) -> Dict[str ,Any]:
    if p not in self._state:
      self._state[p] = {}
    return self._state[p]

class SGDMomentum(MyOptimwithState):
  def __init__(self, params:Iterable[torch.nn.Parameter] , lr:float, momentum:float):
    super().__init__(params , lr)
    self.momentum = momentum

  def _update_pram(self, p:torch.nn.parameter) -> None:
    st = self._state(p)
    v = st.get("v" , 0.0)
    v  = self.momentum * v + self.lr * p.grad
    p += v
    st["v"] = v


In [ ]:
def softmax_naive(x):
	ex = torch.exp(x)
	print(ex)
	print(ex.sum(dim=-1 , keepdim=True))
	return ex / ex.sum(dim=-1 , keepdim=True)

def softmax_stable(x):
	z = x.max(dim =-1 , keepdim=True)
	ex = torch.exp(x - z)
	return ex / ex.sum(dim=-1 , keepdim=True)

from abc import ABC, abstractmethod
from typing import Any, Dict, Iterable
import torch
class Myoptimizer(ABC):
	def __init__(self , params:Iterable[torch.nn.Parameter] ,lr: float):
		self.params = list(params)
		self.lr = lr

	def zero_grad(self):
		for p in self.params:
			p.grad = None

	@torch.no_grad()
	def step(self):
		for p in self.params:
			if p.grad is None:
				continue
			self._update_param(p)

	@abstractmethod
	def _update_param(self, p : torch.nn.Parameter):
		pass

class MySGD(Myoptimizer):
	def _update_param(self, p:torch.nn.Parameter):
		p-=p.grad * self.lr


class MyOptimwithState(Myoptimizer):
	def __init__(self,params,lr):
		super().__init__(params,lr)
		self._state : Dict[torch.nn.Parameter , Dict[str,Any]] = {}

	def state(self,p:torch.nn.Parameter):
		if p not in self._state:
			self._state[p] = {}
		return self._state[p]

class SGDMomentum(MyOptimwithState):
	def __init__(self,params , lr, momentum:float):
		super().__init__(params, lr)
		self.momentum = momentum

	def _update_param(self, p:torch.nn.Parameter):
		st = self.state(p)
		v = st.get("v",0.0)
		v = v * self.momentum + self.lr * p.grad
		p += v
		st["v"] = v

class MYAdam(MyOptimwithState):
	def __init__(self,params,lr,beta1:float = 0.9, beta2:float = 0.999,eps:float = 1e-8):
		super().__init__(params,lr)
		self.beta1 = beta1
		self.beta2 = beta2
		self.eps = eps

	def _update_param(self, p:torch.nn.Parameter):
		state = self.get_state(p)
		v = state.get("v",0)
		m = state.get("m",0)
		t = state.get("t", 0)

		t += 1
		m = self.beta1 * m + (1-self.beta1) * p.grad
		v = self.beta2 * v + (1-self.beta2) * p.grad**2

		m_hat = m / (1-self.beta1**t)
		v_hat = v / (1-self.beta2**t)

		p += -self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

		state["v"] = v
		state["m"] = m
		state['t'] = t

		return p






In [ ]:
data = np.random.normal(size=5)
print(data)

b = data.reshape(1,-1)
c = data.reshape(-1 , 1)
print(b.shape)
print(c.shape)

result = np.abs(b-c)
print(result)

In [ ]:
data_size = 5
dim = 2
data = np.random.normal(size=(data_size, dim))
print(data)

b = data

# **Batch Nomalization**

In [ ]:
import torch
import torch.nn as nn

class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum

        self.gamma = nn.Parameter(torch.ones(1,D))   # scale
        self.beta = nn.Parameter(torch.zeros(1,D))   # shift

        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x:torch.Tensor):  # x: (B, D)
        if self.training:
            # 배치 평균, 분산 계산
            mean = x.mean(dim=0 , keepdim=True) # 1, D
             var = x.var(dim=0, unbiased=False , keepdim=True) # 1, D

            # 러닝 평균 / 분산 업데이트 (지수 이동 평균)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            # Test에서는 러닝 통계 사용
            mean, var = self.running_mean, self.running_var


        x_hat = (x - mean) / torch.sqrt(var + self.eps)

        # scale & shift
        out = self.gamma * x_hat + self.beta
        return out
